In [3]:
from mp_api.client import MPRester
from pymatgen.electronic_structure.core import Spin
import pandas as pd

# ==========================
# Materials Project API Key
# ==========================
API_KEY = "ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h"

# ==========================
# Material ID
# ==========================
material_id = "mp-504"      # Change to mp-504 for CuS

# ==========================
# Download Band Structure
# ==========================
with MPRester("ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h") as mpr:
    bs = mpr.get_bandstructure_by_material_id(material_id)

# ==========================
# Save Fermi Energy
# ==========================
with open(f"{material_id}_Fermi_energy.txt", "w") as f:
    f.write(f"Fermi Energy (eV): {bs.efermi:.6f}")

print("Fermi Energy =", bs.efermi)

# ==========================
# Calculate cumulative k-distance
# ==========================
dist = [0]

for i in range(1, len(bs.kpoints)):
    d = bs.kpoints[i].cart_coords - bs.kpoints[i-1].cart_coords
    dist.append(dist[-1] + (d**2).sum()**0.5)

# ==========================
# Save k-point coordinates
# ==========================
k_df = pd.DataFrame({
    "Index": range(len(bs.kpoints)),
    "Distance": dist,
    "Label": [kp.label if kp.label else "" for kp in bs.kpoints]
})

k_df.to_csv(f"{material_id}_kpoints.csv", index=False)

# ==========================
# Save Spin Up Bands
# ==========================
if Spin.up in bs.bands:

    up = pd.DataFrame(bs.bands[Spin.up].T)
    up.insert(0, "Distance", dist)

    up.to_csv(f"{material_id}_band_up.csv", index=False)

    print("Spin-up bands exported.")

# ==========================
# Save Spin Down Bands
# ==========================
if Spin.down in bs.bands:

    down = pd.DataFrame(bs.bands[Spin.down].T)
    down.insert(0, "Distance", dist)

    down.to_csv(f"{material_id}_band_down.csv", index=False)

    print("Spin-down bands exported.")

print("Done!")

Retrieving ElectronicStructureDoc documents: 100%|██████████| 1/1 [00:00<00:00, 12710.01it/s]


Fermi Energy = 5.49246722
Spin-up bands exported.
Done!


In [5]:
from mp_api.client import MPRester
from pymatgen.electronic_structure.core import Spin
import pandas as pd

# ==========================
# Materials Project API Key
# ==========================
API_KEY = "ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h"

# ==========================
# Material ID
# ==========================
material_id = "mp-504"      # Change as needed mp-18750

# ==========================
# Download DOS
# ==========================
with MPRester("ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h") as mpr:
    dos = mpr.get_dos_by_material_id(material_id)

# ==========================
# Energy values
# ==========================
energy = dos.energies - dos.efermi

# ==========================
# Spin-up DOS
# ==========================
df = pd.DataFrame()
df["Energy (eV)"] = energy

if Spin.up in dos.densities:
    df["DOS_up"] = dos.densities[Spin.up]

if Spin.down in dos.densities:
    # Negative values are convenient for plotting mirrored spin-down DOS
    df["DOS_down"] = -dos.densities[Spin.down]

df.to_csv(f"{material_id}_DOS.csv", index=False)

print("DOS exported successfully.")
print("Fermi Energy =", dos.efermi)

Retrieving ElectronicStructureDoc documents: 100%|██████████| 1/1 [00:00<00:00, 7653.84it/s]


DOS exported successfully.
Fermi Energy = 4.99215075


In [6]:
from mp_api.client import MPRester
from pymatgen.electronic_structure.core import Spin, OrbitalType
import pandas as pd

# ======================================
# Materials Project API Key
# ======================================
API_KEY = "ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h"

# ======================================
# Material ID
# ======================================
material_id = "mp-18750"

# ======================================
# Download Complete DOS
# ======================================
with MPRester("ae5qh1jNMlPeClcwSYDEX1IG5b0lG81h") as mpr:
    dos = mpr.get_dos_by_material_id(material_id)

# ======================================
# Energy referenced to EF
# ======================================
energy = dos.energies - dos.efermi

df = pd.DataFrame()
df["Energy (eV)"] = energy

# ======================================
# Elements present
# ======================================
elements = sorted({str(site.specie) for site in dos.structure})

print("Elements found:", elements)

# ======================================
# Export s, p, d and f PDOS
# ======================================
for element in elements:

    print("Processing", element)

    element_dos = dos.get_element_dos()[element]

    for orb in [OrbitalType.s,
                OrbitalType.p,
                OrbitalType.d,
                OrbitalType.f]:

        try:

            pdos = element_dos.get_spd_dos()[orb]

            if Spin.up in pdos.densities:
                df[f"{element}_{orb.name}_up"] = pdos.densities[Spin.up]

            if Spin.down in pdos.densities:
                df[f"{element}_{orb.name}_down"] = -pdos.densities[Spin.down]

        except KeyError:
            pass

# ======================================
# Total DOS
# ======================================
if Spin.up in dos.densities:
    df["Total_up"] = dos.densities[Spin.up]

if Spin.down in dos.densities:
    df["Total_down"] = -dos.densities[Spin.down]

# ======================================
# Save
# ======================================
filename = f"{material_id}_PDOS.csv"

df.to_csv(filename, index=False)

print("Saved:", filename)
print("Fermi Energy =", dos.efermi)

Retrieving ElectronicStructureDoc documents: 100%|██████████| 1/1 [00:00<00:00, 7256.58it/s]


AttributeError: 'Dos' object has no attribute 'structure'

In [7]:
import mp_api
import pymatgen

print(mp_api.__version__)
print(pymatgen.__version__)

AttributeError: module 'mp_api' has no attribute '__version__'

In [8]:
print(type(dos))

<class 'pymatgen.electronic_structure.dos.Dos'>


In [9]:
from mp_api.client import MPRester

print(type(dos))
print(dir(dos))

<class 'pymatgen.electronic_structure.dos.Dos'>
['REDIRECT', '__add__', '__annotate_func__', '__annotations_cache__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__modify_schema__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_generic_json_schema', '_validate_monty', 'as_dict', 'densities', 'efermi', 'energies', 'from_dict', 'get_cbm_vbm', 'get_densities', 'get_gap', 'get_interpolated_gap', 'get_interpolated_value', 'get_smeared_densities', 'load', 'norm_vol', 'save', 'to_json', 'unsafe_hash', 'validate_monty_v1', 'validate_monty_v2']


In [10]:
print(type(bs))

<class 'pymatgen.electronic_structure.bandstructure.BandStructureSymmLine'>


In [11]:
print(dir(dos))

['REDIRECT', '__add__', '__annotate_func__', '__annotations_cache__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__modify_schema__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_generic_json_schema', '_validate_monty', 'as_dict', 'densities', 'efermi', 'energies', 'from_dict', 'get_cbm_vbm', 'get_densities', 'get_gap', 'get_interpolated_gap', 'get_interpolated_value', 'get_smeared_densities', 'load', 'norm_vol', 'save', 'to_json', 'unsafe_hash', 'validate_monty_v1', 'validate_monty_v2']
